In [ ]:
import urllib.request
filename = 'ratebeer.json'
urllib.request.urlretrieve('https://mcauleylab.ucsd.edu/public_datasets/data/beer/beeradvocate.json.gz', filename)


('ratebeer.json', <http.client.HTTPMessage at 0x7f6ec09d3690>)

In [ ]:
!pip install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.0/135.0 kB 4.8 MB/s eta 0:00:00


In [ ]:
import json, ast, gzip
import dask.dataframe as dd

def clean_beer_dataset(filename, columns):
  #used ChatGPT to adjust the project sample code to open json files.
    output_filename = filename + '.jsonl'
    with gzip.open(filename, 'rt', encoding='utf-8') as infile, open(output_filename, 'w', encoding='utf-8') as outfile:
        for line_num, line in enumerate(infile, 1):
            try:
                parsed = ast.literal_eval(line)
                filtered = {col: parsed.get(col) for col in columns}
                json.dump(filtered, outfile)
                outfile.write('\n')
            except Exception as e:
                print(f"Skipping line {line_num} due to error: {e}")

            if line_num >= 150000:
              break

    df = dd.read_json(output_filename, lines=True)
    df_selected_columns = df[columns].dropna()
    return df_selected_columns

columns = ['beer/name', 'beer/style', 'review/profileName','review/overall']
df_cleaned = clean_beer_dataset("ratebeer.json", columns)

In [ ]:
!head -10 'ratebeer.json.jsonl'
!tail -10 'ratebeer.json.jsonl'

{"beer/name": "Sausa Weizen", "beer/style": "Hefeweizen", "review/profileName": "stcules", "review/overall": "1.5"}
{"beer/name": "Red Moon", "beer/style": "English Strong Ale", "review/profileName": "stcules", "review/overall": "3"}
{"beer/name": "Black Horse Black Beer", "beer/style": "Foreign / Export Stout", "review/profileName": "stcules", "review/overall": "3"}
{"beer/name": "Sausa Pils", "beer/style": "German Pilsener", "review/profileName": "stcules", "review/overall": "3"}
{"beer/name": "Cauldron DIPA", "beer/style": "American Double / Imperial IPA", "review/profileName": "johnmichaelsen", "review/overall": "4"}
{"beer/name": "Caldera Ginger Beer", "beer/style": "Herbed / Spiced Beer", "review/profileName": "oline73", "review/overall": "3"}
{"beer/name": "Caldera Ginger Beer", "beer/style": "Herbed / Spiced Beer", "review/profileName": "Reidrover", "review/overall": "3.5"}
{"beer/name": "Caldera Ginger Beer", "beer/style": "Herbed / Spiced Beer", "review/profileName": "alpineb

In [ ]:
import dask, numpy as np
from dask import bag as db
from collections import defaultdict
def shared_beers_by_user(df_cleaned):
  df = df_cleaned.assign(count=1)

  beer_review_counts = df.groupby('beer/name')['count'].sum()
  person_review_counts = df.groupby('review/profileName')['count'].sum()

  valid_beers = beer_review_counts[beer_review_counts >= 2].index
  valid_reviewers = person_review_counts[person_review_counts >= 4].index

  df_filtered = df[df['beer/name'].isin(valid_beers)]
  df_filtered = df[df['review/profileName'].isin(valid_reviewers)]

  df_filtered['beer/name'] = df_filtered['beer/name'].astype('category').cat.as_known()
  df_filtered['review/profileName'] = df_filtered['review/profileName'].astype('category').cat.as_known()

  table = df_filtered.pivot_table(
        index='review/profileName',
        columns='beer/name',
        values='review/overall',
        aggfunc='mean'
    )
  table_filled = table.fillna(-1)
  beer_user_array = table_filled.to_dask_array(lengths=True)
  beer_names = list(table_filled.columns)
  return beer_user_array, beer_names
beer_user_array, beer_names = shared_beers_by_user(df_cleaned)
shared_beers_by_user(df_cleaned)



(dask.array<truediv-fillna-values, shape=(5701, 6068), dtype=float64, chunksize=(5701, 6068), chunktype=numpy.ndarray>,
 ['"Just One More" Scotch Ale',
  '"Requisite" Imperialistic Stout',
  '"The Wind Cried Mari..." Scottish Heather Ale',
  "'99 Wee Heavy Scotch Ale",
  "'Pooya Porter",
  "'Sconnie Pale Ale",
  "'Sconnie Rustic Trail Amber",
  "'Sconnie Tall Blonde Ale",
  "'Tis The Seasonator",
  '0% Brett Saison De Lente',
  '1 A.M. Ale',
  '1.5 IPA',
  '10 Blocks South',
  '10 Maradó',
  '10 Year Clelebration Ale',
  '100% Barrel Fermented Autumn Maple',
  '100% Brett Autumn Maple',
  '100% Brett IPA',
  '1000 Monkeys Imperial Stout',
  '11 Siècles De Normandie',
  '110K+OT Batch #1 - Der Rauch Gott',
  '110K+OT Batch #2 - I.R.I.S.',
  '110K+OT Batch #2 - I.R.I.S. - Barrel Aged',
  '110K+OT Batch #2 - I.R.I.S. - Cherry Hazelnut Cacao Nib Bourbon Barrel Aged',
  '110K+OT Batch #3 - The Other West Coast IPA',
  '110K+OT Batch #3 - The Other West Coast IPA - Cedar Aged (Humidor Series

In [ ]:
import random
def test_user(beer_user_array):
  rand_row = random.randint(0, beer_user_array.shape[0] - 1)
  user_row = beer_user_array[rand_row, :].compute()
  return rand_row, user_row

index, random_user = test_user(beer_user_array)
print("Random user ratings: ", random_user)


Random user ratings:  [-1. -1. -1. ... -1. -1. -1.]


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def compute_cosine_similarity(u, v):
    i = 0
    total = 0
    square_u = 0
    square_v = 0
    while i < len(u):
      total += u[i] * v[i]
      square_u += u[i] ** 2
      square_v += v[i] ** 2
      i += 1
    total /= (np.sqrt(square_u) * np.sqrt(square_v))
    return total

similar_users = compute_cosine_similarity(beer_user_array, random_user)
print(similar_users)

dask.array<truediv, shape=(6068,), dtype=float64, chunksize=(6068,), chunktype=numpy.ndarray>


In [ ]:
import dask.bag as db
def q1b(beer_user_array, random_user):
    random_user = random_user.reshape(1,-1)
    similarities = cosine_similarity(beer_user_array, random_user).flatten()
    similar_users = list(enumerate(similarities))
    return db.from_sequence(similar_users)

similar_ten = q1b(beer_user_array, random_user).topk(10, key=1).compute()
print(similar_ten)

[(3696, np.float64(1.0000000000000047)), (4436, np.float64(0.9870028578185833)), (4011, np.float64(0.9858111371786709)), (426, np.float64(0.9854460649567547)), (4556, np.float64(0.9854290677171933)), (3206, np.float64(0.9853889517340508)), (2880, np.float64(0.9853721126478351)), (3244, np.float64(0.9845676522404294)), (1523, np.float64(0.9842173626302004)), (3659, np.float64(0.9841728384384403))]


In [ ]:
def similar_users_ratings(beer_user_array, similar_ten):
  binary_ratings = []
  for user in similar_ten:
     i = user[0]
     user_vec = beer_user_array[i, :].compute()
     binary_ratings.append(user_vec)
  return binary_ratings

similar_users_binary = similar_users_ratings(beer_user_array, similar_ten)
print(similar_users_binary)

[array([-1., -1., -1., ..., -1., -1., -1.]), array([-1., -1., -1., ..., -1., -1., -1.]), array([-1., -1., -1., ..., -1., -1., -1.]), array([-1., -1., -1., ..., -1., -1., -1.]), array([-1., -1., -1., ..., -1., -1., -1.]), array([-1., -1., -1., ..., -1., -1., -1.]), array([-1., -1., -1., ..., -1., -1., -1.]), array([-1., -1., -1., ..., -1., -1., -1.]), array([-1., -1., -1., ..., -1., -1., -1.]), array([-1., -1., -1., ..., -1., -1., -1.])]


In [ ]:
import numpy as np

def recommend_beers(similar_users_binary, test_user, beer_names, rating_threshold=4):
    user_clone = test_user.copy()
    tried_indexes = [i for i, rating in enumerate(user_clone) if rating != -1]
    hide_index = random.choice(tried_indexes)
    user_clone[hide_index] = -1

    #Used ChatGPT to develop idea for using vstack and masking
    similar_users_matrix = np.vstack(similar_users_binary)
    masked_matrix = np.ma.masked_equal(similar_users_matrix, -1)
    avg_ratings = np.ma.mean(masked_matrix, axis=0).filled(-1)

    not_tried_mask = (user_clone == -1)
    high_rating_mask = avg_ratings >= rating_threshold

    recommend_mask = np.logical_and(not_tried_mask, high_rating_mask)
    recommend_indexes = np.where(recommend_mask)[0]

    hidden_beer = beer_names[hide_index]
    hidden_beer_rating = avg_ratings[hide_index]
    sorted_indexes = recommend_indexes[np.argsort(avg_ratings[recommend_indexes])[::-1]]
    top_five = sorted_indexes[:5]
    recommended_beers = [beer_names[i] for i in top_five]
    all_recommended_beers = [beer_names[i] for i in recommend_indexes]
    print(f"Hid beer '{hidden_beer}' to test recommendation.")
    print(f" hidden beer rating: {hidden_beer_rating:.1f}")
    return recommended_beers, hidden_beer, all_recommended_beers

recommended_beers, hidden_beer, all_recommended_beers = recommend_beers(similar_users_binary, random_user, beer_names)
print(f" Is the hidden_beer in the recommended beer list? : {hidden_beer in all_recommended_beers}")
print(f" All recommended beers for user: {all_recommended_beers}")
print(f" Top 5 recommended beers for user: {recommended_beers}")


Hid beer 'Mean Manalishi Double I.P.A.' to test recommendation.
 hidden beer rating: 3.5
 Is the hidden_beer in the recommended beer list? : False
 All recommended beers for user: ['Founders Centennial IPA', 'Founders Double Trouble', 'Founders Mister E Froot', 'Jai Alai IPA', 'La Meule']
 Top 5 recommended beers for user: ['Founders Mister E Froot', 'Founders Double Trouble', 'La Meule', 'Jai Alai IPA', 'Founders Centennial IPA']
